# Exploración Inicial:
Realiza una exploración inicial de los datos para identificar posibles problemas, como valores
nulos, atípicos o datos faltantes en las columnas relevantes.
Utiliza funciones de Pandas para obtener información sobre la estructura de los datos, la
presencia de valores nulos y estadísticas básicas de las columnas involucradas.
Une los dos conjuntos de datos de la forma más eficiente.

In [ ]:
pip install -U scikit-learn

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [54]:
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.impute import KNNImputer

In [16]:
df_vuelos = pd.read_csv("./Customer Flight Activity.csv")

In [17]:
df_cliente = pd.read_csv("./Customer Loyalty History.csv")

In [ ]:
df_cliente.head()

In [22]:
df_cliente.info()

<class 'pandas.DataFrame'>
RangeIndex: 16737 entries, 0 to 16736
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Loyalty Number      16737 non-null  int64  
 1   Country             16737 non-null  str    
 2   Province            16737 non-null  str    
 3   City                16737 non-null  str    
 4   Postal Code         16737 non-null  str    
 5   Gender              16737 non-null  str    
 6   Education           16737 non-null  str    
 7   Salary              12499 non-null  float64
 8   Marital Status      16737 non-null  str    
 9   Loyalty Card        16737 non-null  str    
 10  CLV                 16737 non-null  float64
 11  Enrollment Type     16737 non-null  str    
 12  Enrollment Year     16737 non-null  int64  
 13  Enrollment Month    16737 non-null  int64  
 14  Cancellation Year   2067 non-null   float64
 15  Cancellation Month  2067 non-null   float64
dtypes: float64(4), 

In [62]:
# Salary sale como 12499, deberiamos tener 16737, por lo que hay 4238 valores nulos
# 1º le digo que me sume y veo cuanto de estos salarios son nulos
df_cliente["Salary"].isnull().sum()

np.int64(4238)

In [63]:
df_cliente["Salary"].describe()

count     12499.000000
mean      79359.340907
std       34749.691464
min        9081.000000
25%       59246.500000
50%       73455.000000
75%       88517.500000
max      407228.000000
Name: Salary, dtype: float64

In [ ]:
# esta linea de codigo es para quitar el negativo en los valoers numericos de salary 
df_cliente["Salary"] = df_cliente["Salary"].abs()

In [23]:
df_vuelos.info()

<class 'pandas.DataFrame'>
RangeIndex: 405624 entries, 0 to 405623
Data columns (total 10 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Loyalty Number               405624 non-null  int64  
 1   Year                         405624 non-null  int64  
 2   Month                        405624 non-null  int64  
 3   Flights Booked               405624 non-null  int64  
 4   Flights with Companions      405624 non-null  int64  
 5   Total Flights                405624 non-null  int64  
 6   Distance                     405624 non-null  int64  
 7   Points Accumulated           405624 non-null  float64
 8   Points Redeemed              405624 non-null  int64  
 9   Dollar Cost Points Redeemed  405624 non-null  int64  
dtypes: float64(1), int64(9)
memory usage: 30.9 MB


In [ ]:
df_vuelos.head()

In [37]:
df_vuelos.duplicated().sum()

np.int64(1864)

In [ ]:
# Creo una variable llamada duplicados y guardamos ahí únicamente las filas repetidas del DataFrame vuelos
duplicados = df_vuelos[df_vuelos.duplicated()]
duplicados.head(20)

In [ ]:
# esta parte del codigo lo que esta haciendo es detectar todas las columnas de cada fila y devuelve un booleno, en el que
# el parametro keep = false  significa "marca todas las que sean duplicadas", false si la fila es unica 


#.sort_values(by=[...]) ordena las filas del DataFrame duplicados según esas tres columnas, 
# en este orden de prioridad: primero por Loyalty Number, y dentro de cada mismo Loyalty Number, por Year, 
# y dentro de cada mismo Year, por Month.
# Esto es útil porque así las filas duplicadas de un mismo cliente y mismo periodo quedan una al lado de la otra,
#  facilitando verlas y compararlas visualmente.

duplicados = df_vuelos[df_vuelos.duplicated(keep=False)]
duplicados.sort_values(by=["Loyalty Number", "Year", "Month"]).head(30)

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
41,101902,2017,1,0,0,0,0,0.0,0,0
42,101902,2017,1,0,0,0,0,0.0,0,0
16942,101902,2017,2,0,0,0,0,0.0,0,0
16943,101902,2017,2,0,0,0,0,0.0,0,0
33843,101902,2017,3,0,0,0,0,0.0,0,0
33844,101902,2017,3,0,0,0,0,0.0,0,0
101447,101902,2017,7,0,0,0,0,0.0,0,0
101448,101902,2017,7,0,0,0,0,0.0,0,0
185952,101902,2017,12,0,0,0,0,0.0,0,0
185953,101902,2017,12,0,0,0,0,0.0,0,0


In [15]:
df_cliente.duplicated().sum()

np.int64(0)

In [ ]:
# Aqui hacemos la union de datos, lo hacemos con un merge y un right, ya que el df right es es de vuelos, el que
# tiene mas datos, concretamente vuelos tiene 405624 entries frente a los 16737 entries de clientes
# creamos un nuevo df llamada aerolinea
df_aerolinea = df_vuelos.merge(df_cliente, on =["Loyalty Number"], how = "right")
df_aerolinea.sample(5)

In [40]:
df_aerolinea.shape

(405624, 25)

# Limpieza de Datos:
### -  Elimina o trata los valores nulos, si los hay, en las columnas clave para asegurar que los datos estén completos.
### - Verifica la consistencia y corrección de los datos para asegurarte de que los datos se presenten de forma coherente.
### - Realiza cualquier ajuste o conversión necesaria en las columnas (por ejemplo, cambiar tipos de datos) para garantizar la adecuación de los datos para el análisis estadístico.

In [57]:
# gestion de duplicados de df_vuelos, anteriormente nos habian dado (1864)
df_aerolinea.duplicated().sum()

np.int64(1864)

In [ ]:
# Borramos los ducplicados porque estamos viendo pares exactamente iguales
# Mismo: cliente, año, mes, vuelos reservados, vuelos acompañados, distancia, puntos
# No hay ninguna diferencia. Eso significa: La misma observación fue registrada dos veces.

df_aerolinea.drop_duplicates(inplace=True)

In [ ]:
#Se detectaron 1.864 filas duplicadas.

# Tras revisar manualmente las observaciones, se comprobó que existían registros completamente idénticos en todas las variables, 
# incluyendo identificador de cliente, año, mes y actividad de vuelos.

# Debido a que estos registros no aportan nueva información y pueden sesgar el análisis posterior, 
# se procedió a eliminarlos.

df_aerolinea.duplicated().sum()

np.int64(0)

In [61]:
df_aerolinea.shape

(403760, 25)

Ya he borrado duplicados ahora estoy con la gestión de nulos, entonces lo primero que hice fue sacar cuantos nulos hay y cunado veo los apuntes de la profesora me dice que cuando hay una variable para imputar tenemos que ver primero si es categorica o numerica, en este caso es numérica porque es el salario y ya lo he revisado previamente, con un .info.

Entonces me dice que si es numerica y en este caso es un porcentaje alto, estamos hablando de un 

In [51]:
nulos_aerolinea = df_aerolinea.isnull().sum()/df_aerolinea.shape[0]*100
nulos_aerolinea = nulos_aerolinea[nulos_aerolinea>0]
nulos_aerolinea.sort_values(ascending=False)

# Ambas columnas no se modifican porque se entiende que son clientes que no han cancelado su membresia con la compañia por eso da un valor nulo
# con el que si que trabajaremos es con Salary
# Cancellation Year: Año en que el cliente canceló su membresía en el programa de lealtad, si aplica.
# Cancellation Month: Mes en que el cliente canceló su membresía en el programa de lealtad, si aplica.


Cancellation Year     87.657535
Cancellation Month    87.657535
Salary                25.312112
dtype: float64

In [ ]:
# aqui hacemos tecnicas avanzadas de imputacion de los nulos de Salary

knn_imputer = KNNImputer(n_neighbors=4)
cols_numericas = ["euribor3m", "age", "duration"]  # las que tengas
df[cols_numericas] = knn_imputer.fit_transform(df[cols_numericas])